In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import cdist
from scipy.stats import zscore
import itertools
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: تعریف توابع تحلیل (6 تابع - 3 مجموعه)
# ============================================================================

# =====================================================================
# مجموعه ۱: تحلیل همبستگی بیرینگ (Bearing) - 1 تابع
# =====================================================================

def analysis_bearing_correlation_regression(file_path, output_filename1, output_filename2, output_filename3):
    """
    تحلیل همبستگی و رگرسیون برای بیرینگ
    شامل: Rolling Correlation + Linear Regression + Moving Average 24h
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل بیرینگ (همبستگی + رگرسیون + MA)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9357','AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                    'AssetID_9368', 'AssetID_9369', 'AssetID_9370','AssetID_9343', 'AssetID_9344', 'AssetID_9408']
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    additional_values = ['AssetID_9343', 'AssetID_9344']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    df_last_30_days = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    print(f"📅 بازه تحلیل (۳۰ روز آخر): {start_analysis_date} تا {last_date}")
    print(f"📊 تعداد رکوردهای بازه تحلیل: {len(df_last_30_days):,}")
    
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    for target in target_sensors:
        temp_storage = {}
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        results_list.append(target_df_wide[['date', 'AssetID'] + all_features])
    
    df_output2 = pd.concat(results_list, ignore_index=True)
    df_additional = df_cleaned[additional_values].reset_index()
    df_additional_renamed = df_additional.rename(columns={col: f"{col}_Value" for col in additional_values})
    df_output2 = pd.merge(df_output2, df_additional_renamed, on='date', how='left')
    
    df_output1 = df_output2.melt(id_vars=['date', 'AssetID'] + [f"{c}_Value" for c in additional_values], 
                                 value_vars=all_features, var_name='AssetID_correlation', value_name='correlation_value').dropna(subset=['correlation_value'])
    print(f"   ✅ خروجی همبستگی (Long): {len(df_output1):,} رکورد")
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")
    
    print("🔄 مرحله 3: محاسبه رگرسیون و میانگین متحرک ۲۴ ساعته...")
    
    regression_results = []
    target_pairs = list(itertools.combinations(target_sensors, 2))
    
    for s1, s2 in target_pairs:
        X = df_last_30_days[[s1]].values
        y = df_last_30_days[s2].values
        
        model = LinearRegression()
        model.fit(X, y)
        y_pred = model.predict(X)
        
        pair_name = f"{s1}_{s2}"
        temp_reg_df = pd.DataFrame({
            'date': df_last_30_days.index,
            'Sensor_pair': pair_name,
            'Regression_Value': y_pred,
            'AssetID_Value_S1': df_last_30_days[s1].values,
            'AssetID_Value_S2': df_last_30_days[s2].values
        })
        
        temp_reg_df = temp_reg_df.set_index('date')
        temp_reg_df['Regression_MA_24h'] = temp_reg_df['Regression_Value'].rolling(window='24h').mean()
        temp_reg_df = temp_reg_df.reset_index()
        regression_results.append(temp_reg_df)
    
    df_output3 = pd.concat(regression_results, ignore_index=True)
    print(f"   ✅ خروجی رگرسیون: {len(df_output3):,} رکورد")
    
    print("💾 مرحله 4: ذخیره خروجی‌ها...")
    
    try:
        os.makedirs(os.path.dirname(output_filename1), exist_ok=True)
        df_output1.sort_values(by=['date', 'AssetID']).to_excel(output_filename1, index=False)
        df_output2.sort_values(by=['date', 'AssetID']).to_excel(output_filename2, index=False)
        df_output3.sort_values(by=['date', 'Sensor_pair']).to_excel(output_filename3, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل ۱ (همبستگی - Long): {output_filename1}")
        print(f"📂 فایل ۲ (همبستگی - Wide): {output_filename2}")
        print(f"📂 فایل ۳ (رگرسیون + MA): {output_filename3}")
        print(f"\n📊 آمار نهایی:")
        print(f"   فایل ۱: {len(df_output1):,} رکورد")
        print(f"   فایل ۲: {len(df_output2):,} رکورد")
        print(f"   فایل ۳: {len(df_output3):,} رکورد")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


# =====================================================================
# مجموعه ۲: تحلیل‌های همبستگی ژنراتور (Generator) - 2 تابع
# =====================================================================

def analysis_generator_correlation_full(file_path, output_filename1, output_filename2):
    """
    تحلیل همبستگی کامل ژنراتور - کد شماره 1
    شامل: Rolling Correlation + خروجی Long + Wide
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل ژنراتور (همبستگی کامل - Long + Wide)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    target_sensors = all_features
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    print(f"📅 شروع تحلیل Rolling از تاریخ: {start_analysis_date}")
    
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    for target in target_sensors:
        temp_storage = {}
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        results_list.append(target_df_wide[['date', 'AssetID'] + all_features])
    
    df_output2 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")
    
    df_output1 = df_output2.melt(id_vars=['date', 'AssetID'], 
                                 value_vars=all_features, 
                                 var_name='AssetID_correlation', 
                                 value_name='correlation_value').dropna(subset=['correlation_value'])
    print(f"   ✅ خروجی همبستگی (Long): {len(df_output1):,} رکورد")
    
    print("💾 مرحله 3: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename1), exist_ok=True)
        df_output1.to_excel(output_filename1, index=False)
        df_output2.to_excel(output_filename2, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل ۱ (ساختار ردیفی - Long): {output_filename1}")
        print(f"📂 فایل ۲ (ساختار ستونی - Wide): {output_filename2}")
        print(f"\n📊 آمار نهایی:")
        print(f"   فایل ۱: {len(df_output1):,} رکورد")
        print(f"   فایل ۲: {len(df_output2):,} رکورد")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


def analysis_generator_correlation_wide_only(file_path, output_filename2):
    """
    تحلیل همبستگی ژنراتور - فقط خروجی Wide
    شامل: Rolling Correlation + فقط خروجی Wide
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل ژنراتور (همبستگی - فقط Wide)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 
                    'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
    target_sensors = all_features
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    print(f"📅 شروع تحلیل Rolling از تاریخ: {start_analysis_date}")
    
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    for target in target_sensors:
        temp_storage = {}
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        results_list.append(target_df_wide[['date', 'AssetID'] + all_features])
    
    df_output2 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")
    
    print("💾 مرحله 3: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename2), exist_ok=True)
        df_output2.to_excel(output_filename2, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل (ساختار ستونی - Wide): {output_filename2}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_output2):,}")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


# =====================================================================
# مجموعه ۳: تحلیل‌های همبستگی روغن‌کاری (Lubrication) - 3 تابع
# =====================================================================

def analysis_lubrication_correlation_regression(file_path, output_filename1, output_filename2, output_filename3):
    """
    تحلیل همبستگی و رگرسیون برای سیستم روغن‌کاری
    شامل: Rolling Correlation + Linear Regression + Moving Average 24h
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل روغن‌کاری (همبستگی + رگرسیون + MA)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9343','AssetID_9375', 'AssetID_8341', 'AssetID_8342', 'AssetID_8343', 
                    'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    target_sensors = ['AssetID_9343','AssetID_9357','AssetID_9375', 'AssetID_8341', 'AssetID_8343', 
                      'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    df_last_30_days = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    print(f"📅 بازه تحلیل (۳۰ روز آخر): {start_analysis_date} تا {last_date}")
    print(f"📊 تعداد رکوردهای بازه تحلیل: {len(df_last_30_days):,}")
    
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    for target in target_sensors:
        temp_storage = {}
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        results_list.append(target_df_wide[['date', 'AssetID'] + all_features])
    
    df_output1 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output1):,} رکورد")
    
    df_output2 = df_output1.melt(id_vars=['date', 'AssetID'], 
                                 value_vars=all_features, 
                                 var_name='AssetID_correlation', 
                                 value_name='correlation_value').dropna(subset=['correlation_value'])
    print(f"   ✅ خروجی همبستگی (Long): {len(df_output2):,} رکورد")
    
    print("🔄 مرحله 3: محاسبه رگرسیون و میانگین متحرک ۲۴ ساعته...")
    
    regression_results = []
    target_pairs = list(itertools.combinations(target_sensors, 2))
    
    for s1, s2 in target_pairs:
        X = df_last_30_days[[s1]].values
        y = df_last_30_days[s2].values
        
        model = LinearRegression()
        model.fit(X, y)
        y_pred = model.predict(X)
        
        pair_name = f"{s1}_{s2}"
        temp_reg_df = pd.DataFrame({
            'date': df_last_30_days.index,
            'Sensor_pair': pair_name,
            'Regression_Value': y_pred,
            'AssetID_Value_S1': df_last_30_days[s1].values,
            'AssetID_Value_S2': df_last_30_days[s2].values
        })
        
        temp_reg_df = temp_reg_df.set_index('date')
        temp_reg_df['Regression_MA_24h'] = temp_reg_df['Regression_Value'].rolling(window='24h').mean()
        temp_reg_df = temp_reg_df.reset_index()
        regression_results.append(temp_reg_df)
    
    df_output3 = pd.concat(regression_results, ignore_index=True)
    print(f"   ✅ خروجی رگرسیون: {len(df_output3):,} رکورد")
    
    print("💾 مرحله 4: ذخیره خروجی‌ها...")
    
    try:
        os.makedirs(os.path.dirname(output_filename1), exist_ok=True)
        df_output1.sort_values(by=['date', 'AssetID']).to_excel(output_filename1, index=False)
        df_output2.sort_values(by=['date', 'AssetID']).to_excel(output_filename2, index=False)
        df_output3.sort_values(by=['date', 'Sensor_pair']).to_excel(output_filename3, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل ۱ (همبستگی - Wide): {output_filename1}")
        print(f"📂 فایل ۲ (همبستگی - Long): {output_filename2}")
        print(f"📂 فایل ۳ (رگرسیون + MA): {output_filename3}")
        print(f"\n📊 آمار نهایی:")
        print(f"   فایل ۱: {len(df_output1):,} رکورد")
        print(f"   فایل ۲: {len(df_output2):,} رکورد")
        print(f"   فایل ۳: {len(df_output3):,} رکورد")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


def analysis_lubrication_correlation(file_path, output_filename1, output_filename2):
    """
    تحلیل همبستگی برای سیستم روغن‌کاری
    شامل: Rolling Correlation + خروجی Long و Wide
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل روغن‌کاری (همبستگی - Long + Wide)")
    print(f"{'='*60}")
    
    all_features = ['AssetID_9343','AssetID_9357','AssetID_9375', 'AssetID_8341','AssetID_8342',
                    'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']
    target_sensors = all_features
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    before_count = len(df_raw)
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    print(f"📅 شروع تحلیل Rolling از تاریخ: {start_analysis_date}")
    
    print("🔄 مرحله 2: محاسبه ماتریس کوریلیشن متحرک...")
    
    results_list = []
    for target in target_sensors:
        temp_storage = {}
        for feature in all_features:
            if target == feature:
                temp_storage[feature] = np.nan
                continue
            rolling_series = df_cleaned[target].rolling(window='30D').corr(df_cleaned[feature])
            temp_storage[feature] = rolling_series[rolling_series.index >= start_analysis_date]
        
        target_df_wide = pd.DataFrame(temp_storage)
        target_df_wide['AssetID'] = target
        target_df_wide = target_df_wide.reset_index()
        results_list.append(target_df_wide[['date', 'AssetID'] + all_features])
    
    df_output2 = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ خروجی همبستگی (Wide): {len(df_output2):,} رکورد")
    
    df_output1 = df_output2.melt(id_vars=['date', 'AssetID'], 
                                 value_vars=all_features, 
                                 var_name='AssetID_correlation', 
                                 value_name='correlation_value').dropna(subset=['correlation_value'])
    print(f"   ✅ خروجی همبستگی (Long): {len(df_output1):,} رکورد")
    
    print("💾 مرحله 3: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename1), exist_ok=True)
        df_output1.to_excel(output_filename1, index=False)
        df_output2.to_excel(output_filename2, index=False)
        
        print(f"\n✅ عملیات با موفقیت پایان یافت.")
        print(f"📂 فایل ۱ (ساختار ردیفی - Long): {output_filename1}")
        print(f"📂 فایل ۲ (ساختار ستونی - Wide): {output_filename2}")
        print(f"\n📊 آمار نهایی:")
        print(f"   فایل ۱: {len(df_output1):,} رکورد")
        print(f"   فایل ۲: {len(df_output2):,} رکورد")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل‌ها: {e}")
        return False


def analysis_lubrication_partial_correlation_rca(file_path, output_filename3):
    """
    تحلیل همبستگی جزئی و RCA برای سیستم روغن‌کاری
    شامل: Partial Correlation + RCA + Root Cause Analysis
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل روغن‌کاری (همبستگی جزئی + RCA)")
    print(f"{'='*60}")
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ فایل ورودی خوانده شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    numeric_columns = df_raw.select_dtypes(include=[np.number]).columns.tolist()
    if 'date' in numeric_columns:
        numeric_columns.remove('date')
    
    print(f"\n📋 ستون‌های عددی شناسایی شده: {len(numeric_columns)} سنسور")
    if len(numeric_columns) < 2:
        print("❌ خطا: تعداد ستون‌های عددی کافی نیست.")
        return False
    
    all_features = numeric_columns
    
    print("\n🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    unique_labels = set(labels)
    n_clusters = len([x for x in unique_labels if x != -1])
    print(f"   تعداد خوشه‌های یافت شده: {n_clusters}")
    
    before_count = len(df_raw)
    
    if n_clusters == 0:
        z_scores = np.abs(zscore(scaled_data))
        outlier_mask = (z_scores > 3).any(axis=1)
        df_cleaned = df_raw[~outlier_mask].copy()
        df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
        print(f"   پس از حذف outliers با Z-score: {len(df_cleaned):,} رکورد")
    else:
        cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) 
                           for cid in unique_labels if cid != -1}
        
        def calculate_distance(i):
            label, point = labels[i], scaled_data[i].reshape(1, -1)
            if label != -1:
                return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
            return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
        
        df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
        df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
        df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
        print(f"   پس از حذف داده‌های پرت: {len(df_cleaned):,} رکورد")
    
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    print("\n🔄 مرحله 2: جداسازی بازه‌های نرمال و خرابی...")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=30)
    
    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)].copy()
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    
    print(f"   بازه نرمال: {baseline_start} تا {baseline_end} ({len(df_baseline):,} رکورد)")
    print(f"   بازه خرابی: {start_analysis_date} تا {last_date} ({len(df_fault):,} رکورد)")
    
    if len(df_baseline) < 5 or len(df_fault) < 5:
        print("❌ خطا: داده‌های کافی برای تحلیل وجود ندارد.")
        return False
    
    def partial_correlation_matrix(data, columns):
        corr_matrix = data[columns].corr().values
        corr_matrix_regularized = corr_matrix + np.eye(corr_matrix.shape[0]) * 1e-6
        try:
            precision_matrix = np.linalg.inv(corr_matrix_regularized)
        except np.linalg.LinAlgError:
            precision_matrix = np.linalg.pinv(corr_matrix_regularized)
        
        p = precision_matrix.copy()
        diag = np.sqrt(np.diag(p))
        diag[diag == 0] = 1e-6
        partial_corr = -p / np.outer(diag, diag)
        np.fill_diagonal(partial_corr, 1.0)
        partial_corr = np.clip(partial_corr, -1, 1)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)
    
    def normalize_scores(scores_dict):
        values = np.array(list(scores_dict.values()))
        if values.max() == values.min():
            return {k: 0.5 for k in scores_dict.keys()}
        return {k: (v - values.min()) / (values.max() - values.min()) for k, v in scores_dict.items()}
    
    print("\n🔄 مرحله 3: محاسبه ماتریس همبستگی جزئی...")
    partial_corr_baseline = partial_correlation_matrix(df_baseline, all_features)
    partial_corr_fault = partial_correlation_matrix(df_fault, all_features)
    print(f"   ✅ همبستگی جزئی برای {len(all_features)} سنسور محاسبه شد")
    
    print("\n🔄 مرحله 4: محاسبه نمرات RCA...")
    
    deviation_scores = {}
    for col in all_features:
        baseline_mean = df_baseline[col].mean()
        baseline_std = df_baseline[col].std()
        if baseline_std == 0:
            baseline_std = 1e-6
        fault_mean = df_fault[col].mean()
        deviation_scores[col] = abs((fault_mean - baseline_mean) / baseline_std)
    
    change_scores = {}
    for col in all_features:
        changes = [abs(partial_corr_fault.loc[col, other] - partial_corr_baseline.loc[col, other]) 
                   for other in all_features if other != col]
        change_scores[col] = np.mean(changes) if changes else 0
    
    strength_scores = {}
    for col in all_features:
        strengths = [abs(partial_corr_fault.loc[col, other]) for other in all_features if other != col]
        strength_scores[col] = np.mean(strengths) if strengths else 0
    
    deviation_norm = normalize_scores(deviation_scores)
    change_norm = normalize_scores(change_scores)
    strength_norm = normalize_scores(strength_scores)
    
    rca_scores = {}
    for col in all_features:
        rca_scores[col] = (deviation_norm[col] * change_norm[col] * strength_norm[col]) ** (1/3)
    
    rca_normalized = normalize_scores(rca_scores)
    
    rca_summary = pd.DataFrame({
        'Sensor': all_features,
        'Change Normalized': [change_norm[col] for col in all_features],
        'Change Scribe': [change_scores[col] for col in all_features],
        'Deviation Normalized': [deviation_norm[col] for col in all_features],
        'Deviation Scribe': [deviation_scores[col] for col in all_features],
        'Strength Score': [strength_norm[col] for col in all_features],
        'Strength Fault': [strength_scores[col] for col in all_features],
        'RCA Suffocation': ['HIGH' if rca_normalized[col] > 0.7 else 
                            'MEDIUM' if rca_normalized[col] > 0.4 else 
                            'LOW' for col in all_features],
        'Root Cause': [col if rca_normalized[col] > 0.7 else '' for col in all_features]
    })
    
    rca_summary['RCA_Score_Temp'] = [rca_normalized[col] for col in all_features]
    rca_summary = rca_summary.sort_values('RCA_Score_Temp', ascending=False).reset_index(drop=True)
    rca_summary = rca_summary.drop('RCA_Score_Temp', axis=1)
    
    top_sensors = rca_summary.head(8)['Sensor'].tolist()
    partial_corr_fault_filtered = partial_corr_fault.loc[top_sensors, top_sensors]
    
    print("💾 مرحله 5: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename3), exist_ok=True)
        with pd.ExcelWriter(output_filename3, engine='openpyxl') as writer:
            rca_summary.to_excel(writer, sheet_name='RCA Summary', index=False)
            partial_corr_fault_filtered.to_excel(writer, sheet_name='Partial Correlation Fault')
            
            if len(all_features) > 8:
                next_sensors = rca_summary.head(16)['Sensor'].tolist()[8:16]
                if next_sensors:
                    partial_corr_fault_filtered2 = partial_corr_fault.loc[next_sensors, next_sensors]
                    partial_corr_fault_filtered2.to_excel(writer, sheet_name='Partial Correlation Fault 2')
        
        print(f"\n✅ فایل خروجی با موفقیت ذخیره شد: {output_filename3}")
        print(f"\n🔍 نتایج تحلیل همبستگی جزئی برای RCA:")
        print("-"*70)
        print(rca_summary[['Sensor', 'RCA Suffocation', 'Root Cause']].head(10).to_string(index=False))
        print("-"*70)
        
        high_suspicion = rca_summary[rca_summary['RCA Suffocation'] == 'HIGH']['Sensor'].tolist()
        if high_suspicion:
            print(f"🎯 سنسور(های) با بیشترین احتمال Root Cause: {high_suspicion}")
        else:
            print(f"🎯 سنسور(های) با بیشترین احتمال Root Cause: {rca_summary.head(2)['Sensor'].tolist()}")
        return True
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) - 6 وظیفه
# ============================================================================

def get_all_jobs():
    """تعریف تمام ۶ وظیفه تحلیل - 3 مجموعه"""
    jobs = []
    
    # مجموعه ۱: بیرینگ (1 تابع)
    bearing_base = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
    bearing_out_base = r'outputs\G11\dsas_g11_bearings_vibration_temp_correlation_detection\correlation_detection\dsas_g11_bearings_vibration_temp_correlation_detection_output'
    
    jobs.append({
        'name': 'Bearing: Correlation + Regression + MA',
        'function': analysis_bearing_correlation_regression,
        'file_path': bearing_base,
        'output_filename1': f'{bearing_out_base}1.xlsx',
        'output_filename2': f'{bearing_out_base}2.xlsx',
        'output_filename3': f'{bearing_out_base}3.xlsx'
    })
    
    # مجموعه ۲: ژنراتور (2 تابع)
    gen_base = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
    gen_out_base = r'outputs\G11\dsas_g11_generator_bearings_correlation_detection\correlation_detection\dsas_g11_generator_bearings_correlation_detection_output'
    
    jobs.append({
        'name': 'Generator: Correlation Full (Long + Wide)',
        'function': analysis_generator_correlation_full,
        'file_path': gen_base,
        'output_filename1': f'{gen_out_base}1.xlsx',
        'output_filename2': f'{gen_out_base}2.xlsx'
    })
    
    jobs.append({
        'name': 'Generator: Correlation Wide Only',
        'function': analysis_generator_correlation_wide_only,
        'file_path': gen_base,
        'output_filename2': f'{gen_out_base}2.xlsx'
    })
    
    # مجموعه ۳: روغن‌کاری (3 تابع)
    lub_base = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
    lub_out_base = r'outputs\G11\dsas_g11_lubrication_system_correlation_detection\correlation_detection\dsas_g11_lubrication_correlation_detection_output'
    
    jobs.append({
        'name': 'Lubrication: Correlation + Regression + MA',
        'function': analysis_lubrication_correlation_regression,
        'file_path': lub_base,
        'output_filename1': f'{lub_out_base}1.xlsx',
        'output_filename2': f'{lub_out_base}2.xlsx',
        'output_filename3': f'{lub_out_base}3.xlsx'
    })
    
    jobs.append({
        'name': 'Lubrication: Correlation (Long + Wide)',
        'function': analysis_lubrication_correlation,
        'file_path': lub_base,
        'output_filename1': f'{lub_out_base}1.xlsx',
        'output_filename2': f'{lub_out_base}2.xlsx'
    })
    
    jobs.append({
        'name': 'Lubrication: Partial Correlation + RCA',
        'function': analysis_lubrication_partial_correlation_rca,
        'file_path': lub_base,
        'output_filename3': f'{lub_out_base}3.xlsx'
    })
    
    return jobs


def run_all_analyses():
    """اجرای تمام ۶ تحلیل به ترتیب"""
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌های همبستگی (۶ وظیفه)")
    print(f"📅 زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 مجموعه‌ها:")
    print("   1. بیرینگ (Bearing) - ۱ تحلیل (۳ خروجی)")
    print("   2. ژنراتور (Generator) - ۲ تحلیل (۳ خروجی)")
    print("   3. روغن‌کاری (Lubrication) - ۳ تحلیل (۶ خروجی)")
    print("="*80)
    print("📊 مجموع: ۱۲ فایل خروجی")
    print("="*80)
    
    jobs = get_all_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"{'#'*80}")
        
        try:
            # تعیین تعداد آرگومان‌های تابع
            import inspect
            func = job['function']
            sig = inspect.signature(func)
            num_params = len(sig.parameters)
            
            if num_params == 4:  # 3 خروجی
                success = func(job['file_path'], job['output_filename1'], 
                               job['output_filename2'], job['output_filename3'])
            elif num_params == 3:  # 2 خروجی
                success = func(job['file_path'], job['output_filename1'], job['output_filename2'])
            elif num_params == 2:  # 1 خروجی
                success = func(job['file_path'], job['output_filename3'])
            else:
                success = False
                print(f"❌ تعداد پارامترهای نامشخص: {num_params}")
            
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    print("\n📋 جزئیات:")
    for i, r in enumerate(results, 1):
        status = "✅" if r['success'] else "❌"
        print(f"   {i:2d}. {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)"""
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد")
    print("📋 شامل ۳ مجموعه (بیرینگ، ژنراتور، روغن‌کاری) = ۶ تحلیل")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            if current_time in ["16:22", "17:35"]:
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    results = run_all_analyses()
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    time.sleep(60)
            
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه جامع تحلیل همبستگی (۳ مجموعه یکپارچه)")
        print("="*80)
        print("📋 مجموعه‌ها و خروجی‌ها:")
        print("   ┌─────────────────────────────────────────────────────────┐")
        print("   │  مجموعه ۱: بیرینگ (Bearing)          → ۳ خروجی      │")
        print("   │  مجموعه ۲: ژنراتور (Generator)       → ۳ خروجی      │")
        print("   │  مجموعه ۳: روغن‌کاری (Lubrication)   → ۶ خروجی      │")
        print("   └─────────────────────────────────────────────────────────┘")
        print("   مجموع: ۱۲ فایل خروجی")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه جامع تحلیل همبستگی (۳ مجموعه یکپارچه)
📋 مجموعه‌ها و خروجی‌ها:
   ┌─────────────────────────────────────────────────────────┐
   │  مجموعه ۱: بیرینگ (Bearing)          → ۳ خروجی      │
   │  مجموعه ۲: ژنراتور (Generator)       → ۳ خروجی      │
   │  مجموعه ۳: روغن‌کاری (Lubrication)   → ۶ خروجی      │
   └─────────────────────────────────────────────────────────┘
   مجموع: ۱۲ فایل خروجی
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد
📋 شامل ۳ مجموعه (بیرینگ، ژنراتور، روغن‌کاری) = ۶ تحلیل
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-12 16:22:22

🚀 شروع اجرای همه تحلیل‌های همبستگی (۶ وظیفه)
📅 زمان: 2026-07-12 16:22:22
📋 مجموعه‌ها:
   1. بیرینگ (Bearing) - ۱ تحلیل (۳ خروجی)
   2. ژنراتور (Generator) - ۲ تحلیل (۳ خروجی)
   3. روغن‌کاری (Lubrication) - ۳ تحلیل (۶ خروجی)
📊 مجموع: ۱۲ فایل خروجی

##########################################################################